# Geração de Dados Sintéticos — `raw.funcionarios`

**Objetivo:** Popular a tabela `raw.funcionarios` com **150 registros** de funcionários.

**Características dos dados:**
- Nomes brasileiros realistas (prenomes + sobrenomes).
- CPF fictício no formato `XXX.XXX.XXX-XX` (sem validação de dígito verificador).
- Cargos e departamentos alinhados com os códigos de área da tabela fato.
- Salários distribuídos por faixa salarial consistente com o cargo.
- Data de admissão entre 2015 e 2024.
- Tipos de contrato: CLT, PJ, ESTAGIO, TEMPORARIO.

**Reprodutibilidade:** `seed = 42`

In [5]:
# ============================================================
# 1. IMPORTS E CONFIGURAÇÕES
# ============================================================
import pandas as pd
import numpy as np
import hashlib
import uuid
import os
import random
from datetime import datetime, date, timedelta

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

QTD_FUNCIONARIOS = 150
SOURCE_SYSTEM    = 'ERP_CORPORATIVO'
SOURCE_ENTITY    = 'funcionarios'
INGESTION_ID     = str(uuid.uuid4())
INGESTION_TS     = datetime(2026, 1, 10, 10, 30, 0).strftime('%Y-%m-%dT%H:%M:%S.000Z')

print(f'ingestion_id : {INGESTION_ID}')
print(f'ingestion_ts : {INGESTION_TS}')

ingestion_id : 24b1b014-3ee5-47c6-99a5-da744188d73c
ingestion_ts : 2026-01-10T10:30:00.000Z


In [6]:
# ============================================================
# 2. DADOS DE REFERÊNCIA
# ============================================================

PRENOMES_M = [
    'Carlos', 'Roberto', 'Paulo', 'Ricardo', 'Marcos', 'Felipe', 'André',
    'Rodrigo', 'Lucas', 'Diego', 'Gabriel', 'Rafael', 'Thiago', 'Bruno',
    'Gustavo', 'Fernando', 'Eduardo', 'Leandro', 'Vinicius', 'Leonardo',
    'Pedro', 'Henrique', 'Igor', 'Daniel', 'Matheus', 'Victor', 'Renato',
    'Alexandre', 'Fábio', 'Jorge',
]
PRENOMES_F = [
    'Ana', 'Fernanda', 'Juliana', 'Patrícia', 'Sandra', 'Carla', 'Marina',
    'Beatriz', 'Camila', 'Daniela', 'Letícia', 'Larissa', 'Vanessa', 'Amanda',
    'Priscila', 'Renata', 'Aline', 'Mariana', 'Cristiane', 'Luciana',
    'Gabriela', 'Natalia', 'Michele', 'Claudia', 'Simone', 'Adriana',
    'Rosana', 'Tânia', 'Viviane', 'Elaine',
]
SOBRENOMES = [
    'Silva', 'Santos', 'Oliveira', 'Souza', 'Lima', 'Costa', 'Pereira',
    'Ferreira', 'Rodrigues', 'Almeida', 'Nascimento', 'Carvalho', 'Gomes',
    'Martins', 'Rocha', 'Ribeiro', 'Araujo', 'Mendes', 'Barbosa', 'Castro',
    'Melo', 'Cardoso', 'Nunes', 'Teixeira', 'Moraes', 'Correia', 'Ramos',
    'Moreira', 'Dias', 'Pinto', 'Monteiro', 'Freitas', 'Cunha', 'Vieira',
]

# Departamentos que aparecem na tabela fato + departamentos de suporte
# (código, nome completo)
DEPARTAMENTOS = [
    ('COM',  'Comercial'),
    ('COMP', 'Compliance'),
    ('FIN',  'Financeiro'),
    ('JUR',  'Jurídico'),
    ('MKT',  'Marketing'),
    ('OP',   'Operações'),
    ('RH',   'Recursos Humanos'),
    ('TI',   'Tecnologia da Informação'),
    ('ADM',  'Administração'),
    ('CTB',  'Contabilidade'),
]

# Cargo → (nível_salario_min, nível_salario_max)
CARGOS = [
    # Alta liderança
    ('Diretor(a)',                      25000, 50000),
    ('Gerente Sênior',                  18000, 30000),
    # Gerência
    ('Gerente',                         12000, 22000),
    ('Coordenador(a)',                   8000, 15000),
    # Analistas
    ('Analista Sênior',                  7000, 12000),
    ('Analista Pleno',                   5000,  9000),
    ('Analista Júnior',                  3500,  6000),
    # Assistentes / Técnicos
    ('Assistente Administrativo',        2200,  4000),
    ('Técnico(a)',                       3000,  5500),
    ('Especialista',                     6000, 11000),
    # TI
    ('Desenvolvedor(a) Sênior',          9000, 18000),
    ('Desenvolvedor(a) Pleno',           6000, 11000),
    ('Desenvolvedor(a) Júnior',          3500,  6000),
    # Outros
    ('Consultor(a)',                     7000, 14000),
    ('Estagiário(a)',                    1000,  2000),
]

TIPOS_CONTRATO = ['CLT', 'CLT', 'CLT', 'PJ', 'PJ', 'ESTAGIO', 'TEMPORARIO']  # ponderado

print(f'Prenomes M: {len(PRENOMES_M)} | F: {len(PRENOMES_F)}')
print(f'Sobrenomes: {len(SOBRENOMES)}')
print(f'Departamentos: {len(DEPARTAMENTOS)}')
print(f'Cargos: {len(CARGOS)}')

Prenomes M: 30 | F: 30
Sobrenomes: 34
Departamentos: 10
Cargos: 15


In [7]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

def gerar_cpf(n):
    """CPF fictício no formato XXX.XXX.XXX-XX (sem dígito verificador real)."""
    base = str(n).zfill(9)
    d1   = (sum(int(base[i]) * (10 - i) for i in range(9)) % 11)
    d1   = 0 if d1 < 2 else 11 - d1
    d2   = (sum(int(base[i]) * (11 - i) for i in range(9)) + d1 * 2) % 11
    d2   = 0 if d2 < 2 else 11 - d2
    cpf  = f'{base[:3]}.{base[3:6]}.{base[6:9]}-{d1}{d2}'
    return cpf

def gerar_data_admissao(rng_seed):
    """Data aleatória entre 2015-01-01 e 2024-12-31."""
    r    = random.Random(rng_seed)
    inicio = date(2015, 1, 1)
    fim    = date(2024, 12, 31)
    delta  = (fim - inicio).days
    return (inicio + timedelta(days=r.randint(0, delta))).strftime('%Y-%m-%d')

def gerar_hash(row_dict):
    campos = ['id_funcionario_raw', 'nome', 'cpf', 'cargo', 'departamento']
    conteudo = '|'.join(str(row_dict.get(c, '')) for c in campos)
    return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()

print('Funções auxiliares definidas.')

Funções auxiliares definidas.


In [8]:
# ============================================================
# 4. GERAÇÃO DOS FUNCIONÁRIOS
# ============================================================
rng_geral  = np.random.RandomState(SEED)
rng_sal    = np.random.RandomState(SEED + 10)

registros = []

# Distribuição de departamentos: proporções aproximadas de uma empresa real
# Total: 150 funcionários
DISTRIB_DEPTO = {
    'COM':  22,
    'TI':   26,
    'OP':   22,
    'RH':   10,
    'FIN':  17,
    'MKT':  14,
    'JUR':   8,
    'COMP':  8,
    'ADM':  11,
    'CTB':  12,
}
assert sum(DISTRIB_DEPTO.values()) == QTD_FUNCIONARIOS, \
    f'Soma da distribuição = {sum(DISTRIB_DEPTO.values())}, esperado {QTD_FUNCIONARIOS}'

# Mapa de código → nome do departamento
DEPTO_NOME = dict(DEPARTAMENTOS)

# Cargos mais adequados por departamento
CARGOS_POR_DEPTO = {
    'TI':   ['Desenvolvedor(a) Sênior', 'Desenvolvedor(a) Pleno', 'Desenvolvedor(a) Júnior',
              'Analista Sênior', 'Analista Pleno', 'Especialista', 'Gerente', 'Coordenador(a)'],
    'FIN':  ['Analista Sênior', 'Analista Pleno', 'Analista Júnior', 'Gerente',
              'Coordenador(a)', 'Assistente Administrativo', 'Especialista'],
    'RH':   ['Analista Sênior', 'Analista Pleno', 'Analista Júnior', 'Gerente',
              'Coordenador(a)', 'Assistente Administrativo', 'Especialista'],
    'MKT':  ['Analista Sênior', 'Analista Pleno', 'Analista Júnior', 'Gerente',
              'Coordenador(a)', 'Especialista', 'Consultor(a)'],
    'JUR':  ['Especialista', 'Analista Sênior', 'Analista Pleno', 'Gerente Sênior',
              'Consultor(a)', 'Coordenador(a)'],
    'COM':  ['Analista Sênior', 'Analista Pleno', 'Analista Júnior', 'Gerente',
              'Coordenador(a)', 'Consultor(a)', 'Assistente Administrativo'],
    'OP':   ['Técnico(a)', 'Analista Pleno', 'Analista Júnior', 'Coordenador(a)',
              'Assistente Administrativo', 'Gerente', 'Especialista'],
    'COMP': ['Analista Sênior', 'Especialista', 'Gerente Sênior', 'Consultor(a)',
              'Coordenador(a)', 'Analista Pleno'],
    'ADM':  ['Assistente Administrativo', 'Analista Júnior', 'Analista Pleno',
              'Coordenador(a)', 'Gerente', 'Técnico(a)'],
    'CTB':  ['Analista Sênior', 'Analista Pleno', 'Analista Júnior', 'Coordenador(a)',
              'Especialista', 'Gerente', 'Assistente Administrativo'],
}
# Mapa cargo → (sal_min, sal_max)
CARGO_SALARIO = {c[0]: (c[1], c[2]) for c in CARGOS}

seq = 1
for cod_depto, qtd in DISTRIB_DEPTO.items():
    cargos_disponiveis = CARGOS_POR_DEPTO[cod_depto]

    for i in range(qtd):
        # Gênero
        genero = rng_geral.choice(['M', 'F'])
        prenome   = rng_geral.choice(PRENOMES_M if genero == 'M' else PRENOMES_F)
        sobrenome = rng_geral.choice(SOBRENOMES)
        nome      = f'{prenome} {sobrenome}'

        cpf = gerar_cpf(seq * 1000 + 100)

        # Cargo (distribuição: mais analistas/técnicos que diretores)
        cargo = rng_geral.choice(cargos_disponiveis)

        # Salário dentro da faixa do cargo
        sal_min, sal_max = CARGO_SALARIO.get(cargo, (3000, 8000))
        salario = round(rng_sal.uniform(sal_min, sal_max), 2)

        # Tipo de contrato — estagiários ganham menos
        if cargo == 'Estagiário(a)':
            tipo_contrato = 'ESTAGIO'
        elif cargo == 'Consultor(a)':
            tipo_contrato = rng_geral.choice(['PJ', 'PJ', 'CLT'])
        else:
            tipo_contrato = rng_geral.choice(TIPOS_CONTRATO)

        data_admissao = gerar_data_admissao(seq * 3 + 7)

        row = {
            'id_funcionario_raw': seq,
            'nome':               nome,
            'cpf':                cpf,
            'cargo':              cargo,
            'departamento':       cod_depto,
            'data_admissao':      data_admissao,
            'salario':            salario,
            'tipo_contrato':      tipo_contrato,
        }
        row['raw_row_hash'] = gerar_hash(row)
        registros.append(row)
        seq += 1

print(f'Funcionários gerados: {len(registros)}')

Funcionários gerados: 150


In [9]:
# ============================================================
# 5. CONSOLIDAÇÃO E METADADOS
# ============================================================

for seq_meta, row in enumerate(registros, start=1):
    row['ingestion_id']  = INGESTION_ID
    row['ingestion_ts']  = INGESTION_TS
    row['source_system'] = SOURCE_SYSTEM
    row['source_entity'] = SOURCE_ENTITY
    row['row_seq']       = seq_meta

COLUNAS = [
    'id_funcionario_raw', 'nome', 'cpf', 'cargo', 'departamento',
    'data_admissao', 'salario', 'tipo_contrato',
    'ingestion_id', 'ingestion_ts', 'source_system', 'source_entity',
    'row_seq', 'raw_row_hash',
]
df_func = pd.DataFrame(registros, columns=COLUNAS)

print(f'Shape final: {df_func.shape}')
df_func.head()

Shape final: (150, 14)


,id_funcionario_raw,nome,cpf,cargo,departamento,data_admissao,salario,tipo_contrato,ingestion_id,ingestion_ts,source_system,source_entity,row_seq,raw_row_hash
0,1,Leonardo Dias,000.001.100-27,Assistente Administrativo,COM,2021-05-29,3681.60,CLT,24b1b014-3ee5-47c6-99a5-da744188d73c,2026-01-10T10:30:00.000Z,ERP_CORPORATIVO,funcionarios,1,b79e14b4f103e3aea2cf55981364640b6501dcc61464df...
1,2,Viviane Melo,000.002.100-80,Assistente Administrativo,COM,2017-11-26,2247.01,CLT,24b1b014-3ee5-47c6-99a5-da744188d73c,2026-01-10T10:30:00.000Z,ERP_CORPORATIVO,funcionarios,2,0f69e0db81fed1de6c8366eb7b638f000806a15ba92702...
2,3,Igor Nascimento,000.003.100-34,Analista Júnior,COM,2019-01-20,4026.93,PJ,24b1b014-3ee5-47c6-99a5-da744188d73c,2026-01-10T10:30:00.000Z,ERP_CORPORATIVO,funcionarios,3,80a1809f1f0e0de9b9f665fa66a36c0687ea3bede9665d...
3,4,Beatriz Teixeira,000.004.100-98,Analista Júnior,COM,2022-08-05,5046.05,ESTAGIO,24b1b014-3ee5-47c6-99a5-da744188d73c,2026-01-10T10:30:00.000Z,ERP_CORPORATIVO,funcionarios,4,738a6d007739d36d26201c1defbb14b83a30bf4428f241...
4,5,Roberto Teixeira,000.005.100-41,Gerente,COM,2016-07-28,12982.84,ESTAGIO,24b1b014-3ee5-47c6-99a5-da744188d73c,2026-01-10T10:30:00.000Z,ERP_CORPORATIVO,funcionarios,5,131171d5d488e46f18fe4f5eabf6850975e43db9a50365...


In [10]:
# ============================================================
# 6. VALIDAÇÕES
# ============================================================

DEPTOS_FATO = {'COM', 'COMP', 'FIN', 'JUR', 'MKT', 'OP', 'RH', 'TI'}

assert len(df_func) == QTD_FUNCIONARIOS
assert df_func['id_funcionario_raw'].nunique() == QTD_FUNCIONARIOS
assert df_func['cpf'].nunique() == QTD_FUNCIONARIOS, 'CPFs duplicados!'
assert df_func['salario'].min() > 0
assert DEPTOS_FATO.issubset(set(df_func['departamento'])), 'Departamentos da fato ausentes!'

print('✔ 150 funcionários gerados.')
print('✔ CPFs únicos, salários positivos.')
print('✔ Todos os departamentos da tabela fato estão presentes.')
print()
print('Distribuição por departamento:')
print(df_func['departamento'].value_counts().sort_index().to_string())
print()
print('Distribuição por tipo_contrato:')
print(df_func['tipo_contrato'].value_counts().to_string())
print()
print('Estatísticas salariais:')
print(df_func['salario'].describe().round(2).to_string())

✔ 150 funcionários gerados.
✔ CPFs únicos, salários positivos.
✔ Todos os departamentos da tabela fato estão presentes.

Distribuição por departamento:
departamento
ADM     11
COM     22
COMP     8
CTB     12
FIN     17
JUR      8
MKT     14
OP      22
RH      10
TI      26

Distribuição por tipo_contrato:
tipo_contrato
CLT           67
PJ            42
ESTAGIO       22
TEMPORARIO    19

Estatísticas salariais:
count      150.00
mean      8934.25
std       4617.81
min       2247.01
25%       5398.89
50%       8136.72
75%      11251.71
max      24070.33


In [11]:
# ============================================================
# 7. EXPORTAÇÃO PARA CSV
# ============================================================

workspace   = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
output_dir  = os.path.join(workspace, 'data', 'raw', 'funcionarios')
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'funcionarios.csv')
df_func.to_csv(output_path, index=False, encoding='utf-8')

print(f'Arquivo exportado: {output_path}')
print(f'Total de registros: {len(df_func)}')

Arquivo exportado: c:\Users\Adam\Documents\Repositorio\TCC\SCAP\data\raw\funcionarios\funcionarios.csv
Total de registros: 150
